# 4팀 뉴스 점수 만들기 (H3·H4용) — Colab에서 실행

**하는 일:** 드라이브의 빅카인즈 기사(담당 1~4 폴더) → 정리 → 제목으로 회사 매칭 → 유형 분류(공시형·논란형·화제형·기타)
→ **KR-FinBert-SC 감성 점수** → 종목×월 요약 파일 1개.

**결과로 받는 것 (드라이브 `내 드라이브/4팀_뉴스결과/`)**
- `news_monthly.csv` ← **이것만 Claude 채팅에 올리면 됨** (종목·월별 기사 수와 평균 점수, 원문 없음)
- `news_coverage.csv` ← 종목·연도별 기사 수 (수집 누락 확인용, 같이 올리기)
- `type_check_sample200.csv` ← 사람 확인용 무작위 200건(제목 포함 → **드라이브 밖으로 내보내지 말 것**)

**시작 전 준비 (1번만)**
1. 런타임 → 런타임 유형 변경 → **T4 GPU**
2. 공유받은 폴더 `뉴스_원몬_비공개`가 내 드라이브에 없으면: 드라이브 웹에서 폴더 오른쪽 클릭 → **정리 → 바로가기 추가 → 내 드라이브**
3. 위에서부터 셀을 차례로 실행 (중간에 끊겨도 다시 실행하면 이어서 함: 파일별·점수별 캐시)

이용 조건: 모델은 재학습하지 않고 그대로 사용, 원문은 드라이브 밖으로 내보내지 않음.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install python-calamine
import os, glob
# 폴더 위치 자동 찾기 (못 찾으면 아래 NEWS_DIR을 직접 고치기)
cands = glob.glob('/content/drive/MyDrive/**/뉴스_원몬_비공개', recursive=True) + glob.glob('/content/drive/Shareddrives/**/뉴스_원몬_비공개', recursive=True)
NEWS_DIR = cands[0] if cands else '/content/drive/MyDrive/뉴스_원몬_비공개'
OUT_DIR = '/content/drive/MyDrive/4팀_뉴스결과'
CACHE = OUT_DIR + '/cache'
os.makedirs(CACHE, exist_ok=True)
print('뉴스 폴더:', NEWS_DIR, os.path.exists(NEWS_DIR))
files = sorted(glob.glob(NEWS_DIR + '/**/*.xlsx', recursive=True))
SEARCH = [f for f in files if '검색목록' in os.path.basename(f)]
files = [f for f in files if '검색목록' not in os.path.basename(f)]
print('기사 파일', len(files), '개 /', round(sum(os.path.getsize(f) for f in files)/1e9, 2), 'GB')
print('검색목록 파일:', SEARCH)
import collections; print(collections.Counter(f.split('/')[-2] for f in files))

## 1. 처리 함수 (팀 저장소 `연습장/news/news_core.py`와 같은 코드)

In [ ]:
"""뉴스 처리 핵심 함수 (Colab 노트북과 여기 테스트가 같은 코드를 쓴다)

흐름: 빅카인즈 xlsx 읽기 → 분석제외·중복 제거 → 제목으로 회사 매칭 → 유형 분류(키워드 v1)
      → (Colab) KR-FinBert-SC 점수 → 월 배정(마지막 거래일 규칙) → 종목×월 집계
원문(제목·본문)은 드라이브 안에서만 쓰고, 밖으로 내보내는 파일에는 넣지 않는다.
"""
import re, os, glob, warnings
import pandas as pd, numpy as np

# ---------------- 1. 회사 이름 사전 ----------------
def load_names(search_xlsx):
    """검색목록 '회사·옛이름' 시트 → DataFrame(code, name). 현재 이름 + 옛 이름(' OR ' 구분)."""
    x = pd.read_excel(search_xlsx, sheet_name="회사·옛이름", dtype=str)
    rows = []
    for _, r in x.iterrows():
        code = str(r["종목코드"]).strip().zfill(6)
        names = [r["회사명(현재)"]]
        if isinstance(r["옛 이름 (추정, 확인 필요)"], str):
            names += [s.strip() for s in r["옛 이름 (추정, 확인 필요)"].split(" OR ")]
        for n in names:
            if isinstance(n, str) and len(n.strip()) >= 2:
                rows.append((code, n.strip().replace("車", "차")))
    d = pd.DataFrame(rows, columns=["code", "name"]).drop_duplicates()
    amb = d.groupby("name").code.nunique()
    d = d[d.name.isin(amb[amb == 1].index)]            # 두 종목에 같은 이름 → 모호하므로 제외
    return d

def build_matcher(names):
    """긴 이름부터 찾고, 찾은 자리는 지워서 짧은 이름이 그 안에서 다시 잡히지 않게 한다(SK → SK하이닉스 제외).
    영문·숫자·& 이름은 앞뒤에 영문·숫자·&가 붙으면 다른 단어로 본다(KT → KT&G, KTB 제외)."""
    items = sorted(zip(names.name, names.code), key=lambda t: -len(t[0]))
    alts, code_of = [], {}
    for n, c in items:
        esc = re.escape(n)
        if re.search(r"[A-Za-z0-9]", n):
            esc = r"(?<![A-Za-z0-9&])" + esc + r"(?![A-Za-z0-9&])"
        alts.append(esc); code_of[n] = c
    big = re.compile("|".join(alts))          # 한 번에 훑기: 같은 자리에서는 긴 이름이 먼저 잡힘
    def match(title):
        return sorted({code_of[x.group(0)] for x in big.finditer(title)})
    return match

# ---------------- 2. 기사 읽기 ----------------
COLS = {"뉴스 식별자": "nid", "일자": "date", "언론사": "outlet", "제목": "title", "본문": "body", "분석제외 여부": "excl"}
OUTLETS = {"매일경제", "머니투데이", "서울경제", "파이낸셜뉴스", "한국경제"}

def read_one(path):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            d = pd.read_excel(path, usecols=lambda c: c in COLS, dtype=str, engine="calamine")
        except Exception:
            d = pd.read_excel(path, usecols=lambda c: c in COLS, dtype=str)
    d = d.rename(columns=COLS)
    for c in COLS.values():
        if c not in d: d[c] = None
    d["source_file"] = os.path.relpath(path).split(os.sep)[-2] + "/" + os.path.basename(path)
    return d[list(COLS.values()) + ["source_file"]]

def clean(d):
    n0 = len(d)
    d = d[~d.excl.fillna("").str.contains("중복|예외")]                       # 빅카인즈 분석제외 표시
    d = d[d.outlet.isin(OUTLETS)]                                              # 언론사 5곳만
    d["date"] = pd.to_datetime(d.date.astype(str).str[:8], format="%Y%m%d", errors="coerce")
    d = d.dropna(subset=["date"])
    d["title"] = d.title.fillna("").str.replace("車", "차"); d["body"] = d.body.fillna("")
    d = d.drop_duplicates("nid").drop_duplicates(["title", "outlet", "date"])  # OR 묶음 검색끼리 겹친 기사 제거
    return d, n0

# ---------------- 3. 유형 분류 (키워드 v1) ----------------
# 우선순위: 광고 → 공시형 → 논란형 → 화제형 → 기타.  제목 기준(본문은 200자라 잡음이 많음).
KW = {
    "광고": ["[광고]", "[AD]", "[애드]", "협찬", "기획광고", "[PR]"],
    "공시형": ["유상증자", "무상증자", "증자", "감자", "실적", "영업이익", "영업익", "매출", "순이익", "순익", "흑자", "적자",
              "어닝", "배당", "수주", "공급계약", "단일판매", "특징주", "공시", "목표주가", "목표가", "투자의견", "자사주",
              "신용등급", "상한가", "하한가", "신고가", "신저가", "52주", "시총", "시가총액", "주가", "급등", "급락",
              "순매수", "순매도", "코스피", "증시", "IR", "결산", "분기", "주총", "주주총회", "액면분할", "상장"],
    "논란형": ["횡령", "배임", "소송", "리콜", "압수수색", "파업", "과징금", "검찰", "기소", "구속", "갑질", "논란", "의혹",
              "제재", "적발", "담합", "유출", "불매", "고발", "징계", "비자금", "분식", "탈세", "사고", "폭발", "화재",
              "결함", "피소", "벌금", "공정위", "부당", "불공정", "노조"],
    "화제형": ["신사업", "신제품", "출시", "제휴", "협약", "MOU", "업무협약", "투자", "개발", "진출", "공개", "선보", "론칭",
              "런칭", "인수", "합병", "M&A", "특허", "수출", "기술", "공장", "착공", "준공", "신규", "확대", "상생", "협력",
              "파트너", "선정", "수상", "1위", "세계 최초", "국내 최초", "첫", "도입", "가동", "개소", "오픈", "합작", "개관", "획득", "인증", "확보"],
}
TYPES = ["광고", "공시형", "논란형", "화제형"]

def classify(title):
    for t in TYPES:
        if any(k in title for k in KW[t]):
            return t
    return "기타"

# ---------------- 4. 월 배정 ----------------
def assign_month(dates):
    """마지막 거래일(평일 기준 근사, 한국 공휴일 미반영) 당일과 그 이후 기사는 다음 달로.
    이유: t월 수익률은 마지막 거래일 종가까지라, 그날 기사는 t월 말 정보로 보기 어렵다."""
    per = dates.dt.to_period("M")
    last_bday = (dates + pd.offsets.BMonthEnd(0)).dt.normalize()
    return per.where(dates < last_bday, per + 1).astype(str)

# ---------------- 5. 종목×월 집계 ----------------
def aggregate(scored):
    """scored: code, month, type, s(=P긍정−P부정). 출력: 종목×월 요약(원문 없음)."""
    g = scored.groupby(["month", "code"])
    out = g.size().rename("n_all").to_frame()
    for t, k in [("화제형", "topic"), ("공시형", "disc"), ("논란형", "contro"), ("기타", "etc")]:
        sub = scored[scored.type == t].groupby(["month", "code"])
        out[f"n_{k}"] = sub.size()
        out[f"{k}_tone"] = sub.s.mean()
    out["all_tone"] = scored.dropna(subset=["s"]).groupby(["month", "code"]).s.mean()
    sub = scored[scored.type == "화제형"].groupby(["month", "code"])
    out["topic_pos_share"] = sub.s.apply(lambda x: (x > 0).mean())
    for c in [c for c in out if c.startswith("n_")]:
        out[c] = out[c].fillna(0).astype(int)
    return out.reset_index()


## 2. 기사 읽기 (파일마다 캐시 → 끊겨도 이어서)

In [ ]:
import pandas as pd, numpy as np, hashlib, time
names = load_names(SEARCH[0]); match = build_matcher(names)
print('회사 이름', len(names), '개, 종목', names.code.nunique(), '개')
parts = []; t0 = time.time(); n_read = 0
for i, f in enumerate(files):
    key = hashlib.md5((f + str(os.path.getsize(f))).encode()).hexdigest()[:12]
    cp = f'{CACHE}/m_{key}.parquet'
    if os.path.exists(cp):
        d = pd.read_parquet(cp)
    else:
        d = read_one(f)
        d['n_file'] = len(d)
        d['codes'] = d.title.fillna('').str.replace('車', '차').map(match)
        d = d[d.codes.str.len() > 0]          # 제목에 회사 이름이 있는 기사만 남겨 메모리 절약
        d.to_parquet(cp, index=False)
    parts.append(d)
    if i % 10 == 0: print(i, '/', len(files), f.split('/')[-2:], len(d), f'{time.time()-t0:.0f}s')
raw = pd.concat(parts, ignore_index=True)
art, n0 = clean(raw)
print(f'제목에 회사 이름이 있는 기사 {n0:,} → 분석제외·언론사·중복 제거 후 {len(art):,}')

## 3. 제목으로 회사 매칭 + 유형 분류 + 월 배정

In [ ]:
m = art.explode('codes').dropna(subset=['codes']).rename(columns={'codes': 'code'})
m = m[(m.date >= '2000-12-01') & (m.date <= '2021-12-31')]
m['type'] = m.title.map(classify)
m['month'] = assign_month(m.date)
m = m[m.month <= '2021-12']
print(f'제목에 회사 이름이 있는 기사(종목별) {len(m):,}건, 종목 {m.code.nunique()}개')
print(m.type.value_counts(normalize=True).round(3))
# 사람 확인용 무작위 200건 (교수님 피드백 ②: AI 분류를 사람이 확인)
smp = m.sample(200, random_state=1109)[['code', 'date', 'outlet', 'title', 'type']].assign(사람분류_1='', 사람분류_2='', 메모='')
smp.to_csv(OUT_DIR + '/type_check_sample200.csv', index=False, encoding='utf-8-sig')

## 4. KR-FinBert-SC 감성 점수 (GPU, 기사별 캐시)
점수 s = P(긍정) − P(부정), −1~+1. 제목 + 본문 200자를 넣는다. 광고는 점수를 매기지 않는다.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
MODEL = 'snunlp/KR-FinBert-SC'
tok = AutoTokenizer.from_pretrained(MODEL)
mdl = AutoModelForSequenceClassification.from_pretrained(MODEL).eval()
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; mdl.to(dev)
if dev == 'cuda': mdl.half()
lab = {v.lower(): k for k, v in mdl.config.id2label.items()}; print(dev, mdl.config.id2label)
SC = OUT_DIR + '/scores_by_nid.parquet'
done = pd.read_parquet(SC) if os.path.exists(SC) else pd.DataFrame(columns=['nid', 's'])
todo = m[(m.type != '광고')].drop_duplicates('nid')
todo = todo[~todo.nid.isin(done.nid)]
print('점수 매길 기사', len(todo), '건 (이미 있음', len(done), ')')
texts = (todo.title + ' ' + todo.body.str[:200]).tolist(); ids = todo.nid.tolist()
BS = 256; buf = []; t0 = time.time()
for i in range(0, len(texts), BS):
    enc = tok(texts[i:i+BS], padding=True, truncation=True, max_length=160, return_tensors='pt').to(dev)
    with torch.no_grad():
        p = torch.softmax(mdl(**enc).logits.float(), -1).cpu().numpy()
    buf.append(pd.DataFrame({'nid': ids[i:i+BS], 's': p[:, lab['positive']] - p[:, lab['negative']]}))
    if (i // BS) % 200 == 199 or i + BS >= len(texts):
        done = pd.concat([done] + buf, ignore_index=True); buf = []; done.to_parquet(SC, index=False)
        print(f'{i+BS:,}/{len(texts):,}  {time.time()-t0:.0f}s')

## 5. 종목×월 요약 저장 (원문 없음 → 채팅에 올려도 됨)

In [ ]:
sc = pd.read_parquet(SC)
scored = m.merge(sc, on='nid', how='left')
mon = aggregate(scored)
mon.to_csv(OUT_DIR + '/news_monthly.csv', index=False, encoding='utf-8-sig')
cov = scored.assign(year=scored.month.str[:4]).groupby(['code', 'year']).size().unstack(fill_value=0)
cov.to_csv(OUT_DIR + '/news_coverage.csv', encoding='utf-8-sig')
print(mon.shape, '종목', mon.code.nunique(), '개월', mon.month.nunique())
print(mon[['n_all','n_topic','n_disc','n_contro','topic_tone','all_tone']].describe().round(3))
print('저장 위치:', OUT_DIR)
print('→ news_monthly.csv, news_coverage.csv 두 파일을 Claude 채팅에 올려 주세요.')